# GWAS Tutorial


This notebook is an sgkit port of Hial GWAS tutorial, which deonstrates how to run a genome-wide SNP association test.

In [1]:
import sgkit as sg

import numpy as np
import pandas as pd
import xarray as xr
xr.set_options(display_expand_attrs=False, display_expand_data_vars=True);

In [ ]:
%%bash
bgz_file="/home/sukui/01.data/03.raw_data/1kg/ALL.chr22.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz"
idf_file="/home/sukui/01.data/03.raw_data/1kg/ALL.chr22.icf"
vcz_file="/home/sukui/01.data/03.raw_data/1kg/ALL.chr22.vcz"
vcf2zarr explode --force idf_file
vcf2zarr encode --force -s ALL.chr22.schema.json vcz_file

we used the `vcf2zarr explode` command to first convert the VCF to "intermediate columnar format" (ICF), then the `vcf2zarr encode` command to convert the ICF to Zarr, which by convention is stored in a directory with a `vcz` extension.

Note that we specified a JSON schema file that was created with the `vcf2zarr mkschema` command, then edited to drop some fields that are not needed for this tutorial (sucha as `FORMAT/RT`). It was also update to change the `call_AD` field's third dimension to be `alleles`, which was not set by `vcf2zarr` since the dataset we are using defines `FORMAT/AD` as `.` which means "unknown", rather than R

## Getting to know our data

To start with we'll look at some summary data from the dataset

The simplest thing is to look at the dimensions and data variables in Xarray dataset.

In [2]:
ds = sg.load_dataset("/home/sukui/01.data/03.raw_data/1kg/ALL.chr22.vcz")

In [3]:
ds

<xarray.Dataset> Size: 14GB
Dimensions:                (variants: 1103547, samples: 2504, ploidy: 2,
                            contigs: 86, filters: 1, region_index_values: 1104,
                            region_index_fields: 6, alt_alleles: 8,
                            INFO_CIEND_dim: 2, INFO_CIPOS_dim: 2,
                            INFO_MC_dim: 4, INFO_MEINFO_dim: 4, INFO_VT_dim: 2,
                            alleles: 9)
Dimensions without coordinates: variants, samples, ploidy, contigs, filters,
                                region_index_values, region_index_fields,
                                alt_alleles, INFO_CIEND_dim, INFO_CIPOS_dim,
                                INFO_MC_dim, INFO_MEINFO_dim, INFO_VT_dim,
                                alleles
Data variables: (12/44)
    call_genotype          (variants, samples, ploidy) int8 6GB dask.array<chunksize=(1000, 2504, 2), meta=np.ndarray>
    call_genotype_mask     (variants, samples, ploidy) bool 6GB dask.array<chunksize=(1000, 2504, 2), meta=np.ndarray>
    call_genotype_phased   (variants, samples) bool 3GB dask.array<chunksize=(1000, 2504), meta=np.ndarray>
    contig_id              (contigs) object 688B dask.array<chunksize=(86,), meta=np.ndarray>
    contig_length          (contigs) float64 688B dask.array<chunksize=(86,), meta=np.ndarray>
    filter_description     (filters) object 8B dask.array<chunksize=(1,), meta=np.ndarray>
    ...                     ...
    variant_filter         (variants, filters) bool 1MB dask.array<chunksize=(1000, 1), meta=np.ndarray>
    variant_id             (variants) object 9MB dask.array<chunksize=(1000,), meta=np.ndarray>
    variant_id_mask        (variants) bool 1MB dask.array<chunksize=(1000,), meta=np.ndarray>
    variant_length         (variants) int32 4MB dask.array<chunksize=(1000,), meta=np.ndarray>
    variant_position       (variants) int32 4MB dask.array<chunksize=(1000,), meta=np.ndarray>
    variant_quality        (variants) float32 4MB dask.array<chunksize=(1000,), meta=np.ndarray>
Attributes: (3)

Next we'll use `display_genotypes` to show the first and last few variants and samples

In [8]:
ds["variant_contig_name"] = ds.contig_id[ds.variant_contig]
# ds2 = ds.set_index({"variants1": ("variant_position", "variant_id")})
sg.display_genotypes(ds, max_variants=10, max_samples=5)

samples,HG00096,HG00097,...,NA21143,NA21144
variants,,,,,
0,0|0,0|0,...,0|0,0|0
1,0|0,0|0,...,0|0,0|0
2,0|0,0|0,...,0|0,0|0
3,0|0,0|0,...,0|0,0|0
4,0|0,0|0,...,0|0,0|0
...,...,...,...,...,...
6,0|0,0|0,...,0|0,0|0
7,0|0,0|0,...,0|0,0|0
8,0|0,0|0,...,0|0,0|0


In [9]:
ds.sample_id.values

array(['HG00096', 'HG00097', 'HG00099', ..., 'NA21142', 'NA21143',
       'NA21144'], dtype=object)

In [ ]:
sample_gt = ds.call_genotype[0,:,:]
sample_gt

<xarray.DataArray 'call_genotype' (ploidy: 2)> Size: 2B
dask.array<getitem, shape=(2,), dtype=int8, chunksize=(2,), chunktype=numpy.ndarray>
Dimensions without coordinates: ploidy
Attributes: (1)

In [15]:
sample_gt.values.shape

(2504, 2)

In [3]:
# subset the entire dataset to the first 10 variants/samples
ds.isel(variants=slice(10), samples=slice(10))

<xarray.Dataset> Size: 33kB
Dimensions:                (variants: 10, samples: 10, ploidy: 2, contigs: 86,
                            filters: 1, region_index_values: 1104,
                            region_index_fields: 6, alt_alleles: 8,
                            INFO_CIEND_dim: 2, INFO_CIPOS_dim: 2,
                            INFO_MC_dim: 4, INFO_MEINFO_dim: 4, INFO_VT_dim: 2,
                            alleles: 9)
Dimensions without coordinates: variants, samples, ploidy, contigs, filters,
                                region_index_values, region_index_fields,
                                alt_alleles, INFO_CIEND_dim, INFO_CIPOS_dim,
                                INFO_MC_dim, INFO_MEINFO_dim, INFO_VT_dim,
                                alleles
Data variables: (12/44)
    call_genotype          (variants, samples, ploidy) int8 200B dask.array<chunksize=(10, 10, 2), meta=np.ndarray>
    call_genotype_mask     (variants, samples, ploidy) bool 200B dask.array<chunksize=(10, 10, 2), meta=np.ndarray>
    call_genotype_phased   (variants, samples) bool 100B dask.array<chunksize=(10, 10), meta=np.ndarray>
    contig_id              (contigs) object 688B dask.array<chunksize=(86,), meta=np.ndarray>
    contig_length          (contigs) float64 688B dask.array<chunksize=(86,), meta=np.ndarray>
    filter_description     (filters) object 8B dask.array<chunksize=(1,), meta=np.ndarray>
    ...                     ...
    variant_filter         (variants, filters) bool 10B dask.array<chunksize=(10, 1), meta=np.ndarray>
    variant_id             (variants) object 80B dask.array<chunksize=(10,), meta=np.ndarray>
    variant_id_mask        (variants) bool 10B dask.array<chunksize=(10,), meta=np.ndarray>
    variant_length         (variants) int32 40B dask.array<chunksize=(10,), meta=np.ndarray>
    variant_position       (variants) int32 40B dask.array<chunksize=(10,), meta=np.ndarray>
    variant_quality        (variants) float32 40B dask.array<chunksize=(10,), meta=np.ndarray>
Attributes: (3)

In [5]:
# subset to specific set of variables
ds[["variant_allele", "call_genotype"]]

<xarray.Dataset> Size: 6GB
Dimensions:         (variants: 1103547, alleles: 9, samples: 2504, ploidy: 2)
Dimensions without coordinates: variants, alleles, samples, ploidy
Data variables:
    variant_allele  (variants, alleles) object 79MB dask.array<chunksize=(1000, 9), meta=np.ndarray>
    call_genotype   (variants, samples, ploidy) int8 6GB dask.array<chunksize=(1000, 2504, 2), meta=np.ndarray>
Attributes: (3)

In [7]:
# extract  single variable
ds.call_genotype[:3, :3]

<xarray.DataArray 'call_genotype' (variants: 3, samples: 3, ploidy: 2)> Size: 18B
dask.array<getitem, shape=(3, 3, 2), dtype=int8, chunksize=(3, 3, 2), chunktype=numpy.ndarray>
Dimensions without coordinates: variants, samples, ploidy
Attributes: (1)

In [8]:
# asses the array underlying a single variable
ds.call_genotype.data[:3, :3]

dask.array<getitem, shape=(3, 3, 2), dtype=int8, chunksize=(3, 3, 2), chunktype=numpy.ndarray>

In [15]:
# get a single item from an array as a python scalar
ds.sample_id


<xarray.DataArray 'sample_id' (samples: 2504)> Size: 20kB
dask.array<open_dataset-sample_id, shape=(2504,), dtype=object, chunksize=(2504,), chunktype=numpy.ndarray>
Dimensions without coordinates: samples

Larger subsets of data can be visualized and/or summarized through various sgkit utilities as well as the pandas integration.

In [16]:
ds = sg.simulate_genotype_call_dataset(
    n_variant=1000,
    n_sample=250, 
    missing_pct=0.1
)
sg.display_genotypes(ds, max_variants=8, max_samples=8)

samples,S0,S1,S2,S3,...,S246,S247,S248,S249
variants,,,,,,,,,
0,0/0,1/0,1/0,0/1,...,1/1,0/0,1/1,1/0
1,0/0,0/1,1/1,1/1,...,0/1,1/0,0/.,0/0
2,1/1,0/1,1/0,0/1,...,0/0,1/1,0/1,./1
3,1/0,1/0,1/0,1/0,...,1/1,1/0,0/1,1/.
...,...,...,...,...,...,...,...,...,...
996,0/.,1/1,1/1,0/0,...,./.,0/0,1/0,./0
997,0/0,./0,1/0,1/1,...,1/.,1/0,1/1,./1
998,0/1,0/1,1/1,1/0,...,0/0,0/0,0/1,./1
999,1/0,0/1,1/0,1/1,...,0/1,1/0,0/0,0/1


In [25]:
# A naive version of the above is also possible using only pandas 
# and illustrate the flexibility that comes from being able to transition 
# into and out of array/dataframe representations easily
ds.call_genotype[:5,:5].to_series().unstack().where(lambda df: df >=0, None).fillna('.').astype(str).apply('/'.join,axis=1).unstack()

samples,0,1,2,3,4
variants,,,,,
0,0.0/0,1.0/0,1.0/0,0.0/1,./0
1,0.0/0,0.0/1,1.0/1,1.0/1,./0
2,1.0/1,0.0/1,1.0/0,0.0/1,1.0/0
3,1.0/0,1.0/0,1.0/0,1.0/0,1.0/1
4,./1,./1,1.0/1,0.0/0,0.0/1


In [28]:
# show call rate distribution for each variant using pandas

df = ~ds.call_genotype_mask.to_dataframe()
df.head()

call_genotype_mask
variants samples ploidy                    
0        0       0                     True
                 1                     True
         1       0                     True
                 1                     True
         2       0                     True

In [ ]:
call_rate = df.groupby('variants').mean() # 每个SNP位点的检出率
call_rate

,call_genotype_mask
variants,
0,0.898
1,0.910
2,0.926
3,0.894
4,0.892
...,...
995,0.892
996,0.906
997,0.900


In [34]:
call_rate

,call_genotype_mask
variants,
0,0.898
1,0.910
2,0.926
3,0.894
4,0.892
...,...
995,0.892
996,0.906
997,0.900


## Chunked arrays

Chunked arrays are required when working on large dataset. Libraries for managing chunked arrays such as `Dask Array` and `Zarr` make it possible to implement blockwise algorithms that operate on subsets of arrays without ever requiring them to fit entirely in memory.

By design, they behave almost identically to in-memory rrays within Xarray and can be interchanged freely when provided to sgkit functions. The most notable difference in behavior though is that operations on chunked arrays are evaluated lazily. This means that if an Xarray `Dataset` contains only chunked arrays, no actual computations will be performed until one of the following occurs:

- `Dataset.compute` is called
- `DataArray.compute` is called



In [35]:
import sgkit as sg

ds = sg.simulate_genotype_call_dataset(n_variant=100, n_sample=50, missing_pct=0.1)
ds = ds.chunk(chunks=50)

ds

<xarray.Dataset> Size: 22kB
Dimensions:             (contigs: 1, variants: 100, alleles: 2, samples: 50,
                         ploidy: 2)
Dimensions without coordinates: contigs, variants, alleles, samples, ploidy
Data variables:
    contig_id           (contigs) <U1 4B dask.array<chunksize=(1,), meta=np.ndarray>
    variant_contig      (variants) int64 800B dask.array<chunksize=(50,), meta=np.ndarray>
    variant_position    (variants) int64 800B dask.array<chunksize=(50,), meta=np.ndarray>
    variant_allele      (variants, alleles) |S1 200B dask.array<chunksize=(50, 2), meta=np.ndarray>
    sample_id           (samples) <U3 600B dask.array<chunksize=(50,), meta=np.ndarray>
    call_genotype       (variants, samples, ploidy) int8 10kB dask.array<chunksize=(50, 50, 2), meta=np.ndarray>
    call_genotype_mask  (variants, samples, ploidy) bool 10kB dask.array<chunksize=(50, 50, 2), meta=np.ndarray>
Attributes: (2)

In [36]:
ds.variant_position

<xarray.DataArray 'variant_position' (variants: 100)> Size: 800B
dask.array<xarray-variant_position, shape=(100,), dtype=int64, chunksize=(50,), chunktype=numpy.ndarray>
Dimensions without coordinates: variants
Attributes: (1)

In [ ]:
# call compute via the dask.array api
ds.variant_position.data.compute()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84,
       85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99])

In [38]:
# Coerce to numpy via Xarray
ds.variant_position.values[:5]

array([0, 1, 2, 3, 4])

In [39]:
# compute without unboxing from xarray.DataArray

ds.variant_position.compute()[:5]

<xarray.DataArray 'variant_position' (variants: 5)> Size: 40B
array([0, 1, 2, 3, 4])
Dimensions without coordinates: variants
Attributes: (1)

In [49]:
ds = sg.simulate_genotype_call_dataset(n_variant=1000, n_sample=250, n_contig=23, missing_pct=.9)
ds

<xarray.Dataset> Size: 1MB
Dimensions:             (contigs: 23, variants: 1000, alleles: 2, samples: 250,
                         ploidy: 2)
Dimensions without coordinates: contigs, variants, alleles, samples, ploidy
Data variables:
    contig_id           (contigs) <U2 184B '0' '1' '2' '3' ... '20' '21' '22'
    variant_contig      (variants) int64 8kB 0 0 0 0 0 0 0 ... 22 22 22 22 22 22
    variant_position    (variants) int64 8kB 0 1 2 3 4 5 6 ... 37 38 39 40 41 42
    variant_allele      (variants, alleles) |S1 2kB b'G' b'A' b'T' ... b'A' b'T'
    sample_id           (samples) <U4 4kB 'S0' 'S1' 'S2' ... 'S248' 'S249'
    call_genotype       (variants, samples, ploidy) int8 500kB -1 -1 1 ... -1 -1
    call_genotype_mask  (variants, samples, ploidy) bool 500kB True ... True
Attributes: (2)

In [ ]:
sg.cou

<xarray.DataArray 'call_genotype' (variants: 1000, samples: 250, ploidy: 2)> Size: 500kB
array([[[-1, -1],
        [ 1, -1],
        [-1, -1],
        ...,
        [ 0, -1],
        [-1, -1],
        [-1, -1]],

       [[ 0, -1],
        [-1,  1],
        [-1, -1],
        ...,
        [-1, -1],
        [-1, -1],
        [-1, -1]],

       [[-1,  1],
        [-1, -1],
        [-1, -1],
        ...,
...
        ...,
        [-1, -1],
        [-1, -1],
        [-1, -1]],

       [[-1, -1],
        [ 0, -1],
        [-1, -1],
        ...,
        [-1, -1],
        [-1, -1],
        [-1, -1]],

       [[-1, -1],
        [-1, -1],
        [-1, -1],
        ...,
        [-1, -1],
        [-1,  0],
        [-1, -1]]], dtype=int8)
Dimensions without coordinates: variants, samples, ploidy
Attributes: (2)